# Wanderbricks — Explore Gold Features

Inspect the feature table that feeds the models: null rates, feature-target correlation, and per-station feature plots.

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "iraonfridays")
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
print(f"target: {CATALOG}.{SCHEMA}")

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col

def table_exists(name: str) -> bool:
    return spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{name}")

for t in ["gsod_bronze", "gsod_silver", "weather_features", "forecasts"]:
    print(f"{t:20s} {'EXISTS' if table_exists(t) else 'MISSING - run the DLT pipeline first'}")

In [ ]:
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.functions import col

if not table_exists("weather_features"):
    print("weather_features missing - run the DLT pipeline first")
else:
    feats = spark.table(f"{CATALOG}.{SCHEMA}.weather_features")
    print(f"rows: {feats.count()}")
    display(feats.limit(20))

## Null rates per feature

In [ ]:
if table_exists("weather_features"):
    feats = spark.table(f"{CATALOG}.{SCHEMA}.weather_features")
    feature_cols = [c for c in feats.columns if c not in ("station", "date", "temp_c")]
    nulls = [
        F.count(F.when(col(c).isNull(), 1)).alias(c)
        for c in feature_cols
    ]
    display(
        feats.agg(F.count("*").alias("rows"), *nulls).toPandas().T.rename(
            columns={0: "null_count"})
    )

## Correlation of features with target (`temp_c`)

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

if table_exists("weather_features"):
    feats = spark.table(f"{CATALOG}.{SCHEMA}.weather_features")
    feature_cols = [c for c in feats.columns if c not in ("station", "date")]
    df = feats.select(feature_cols).dropna()
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    vec = assembler.transform(df).select("features")
    corr = Correlation.corr(vec, "features").collect()[0][0].toArray()
    corr_df = pd.DataFrame(corr, index=feature_cols, columns=feature_cols)
    display(corr_df.round(3))

## Per-station feature plot

In [ ]:
import matplotlib.pyplot as plt

if table_exists("weather_features"):
    station = dbutils.widgets.get("station").strip()
    if not station:
        station = (
            spark.table(f"{CATALOG}.{SCHEMA}.gsod_silver")
            .groupBy("station").count().orderBy(F.desc("count"))
            .limit(1).collect()[0][0]
        )
    pdf = (
        spark.table(f"{CATALOG}.{SCHEMA}.weather_features")
        .filter(F.col("station") == station)
        .select("date", "temp_c", "temp_lag_1", "temp_rollmean_7", "temp_rollmean_14")
        .orderBy("date")
        .toPandas()
        .set_index("date")
    )
    print(f"station: {station}")
    pdf.plot(figsize=(14, 5))
    plt.title(f"Weather features - {station}")
    plt.ylabel("temperature °C")
    plt.show()

## What to look for

- **Lags should be highly correlated** with `temp_c` (weather persists day to day).
- **Rolling means** smooth the noise; they are the prior-window average, strictly before the target day.
- Nulls come from missing source readings (sentinel -> NULL in silver) — `05_train_xgb.py` drops them.